In [9]:
import os
import zipfile
import requests
import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [10]:
url = "https://victorzhou.com/movie-reviews-dataset.zip" 
zip_path = "movie_reviews.zip" 
extract_dir = "movie_reviews" 

In [11]:
if not os.path.exists(zip_path): 
    with open(zip_path, "wb") as f: 
        f.write(requests.get(url).content)

In [12]:
if not os.path.exists(extract_dir): 
    with zipfile.ZipFile(zip_path, 'r') as zip_ref: 
        zip_ref.extractall(extract_dir) 

In [13]:
def load_reviews(folder): 
    texts = [] 
    for filename in os.listdir(folder): 
        with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f: 
            texts.append(f.read()) 
    return texts 

In [19]:
train_pos = load_reviews("movie_reviews/movie-reviews-dataset/train/pos")
train_neg = load_reviews("movie_reviews/movie-reviews-dataset/train/neg")

test_pos = load_reviews("movie_reviews/movie-reviews-dataset/test/pos")
test_neg = load_reviews("movie_reviews/movie-reviews-dataset/test/neg")

In [21]:
pos_texts = train_pos + test_pos
neg_texts = train_neg + test_neg

In [22]:
texts = pos_texts + neg_texts 
labels = [1] * len(pos_texts) + [0] * len(neg_texts) 

In [23]:
vocab_size = 10000 
max_len = 200 

In [24]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>") 
tokenizer.fit_on_texts(texts) 
sequences = tokenizer.texts_to_sequences(texts) 
padded = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post') 

In [25]:
X_train, X_test, y_train, y_test = train_test_split(padded, labels, test_size=0.2, random_state=42) 

In [26]:
model = Sequential([ 
            Embedding(vocab_size, 64, input_length=max_len), 
            SimpleRNN(64), 
            Dense(1, activation='sigmoid') 
        ]) 

In [27]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy']) 

In [28]:
model.fit(np.array(X_train), np.array(y_train), epochs=5, batch_size=32, validation_split=0.2) 

Epoch 1/5
1000/1000 [==============================] - 28s 26ms/step - loss: 0.6956 - accuracy: 0.5025 - val_loss: 0.6951 - val_accuracy: 0.5000
Epoch 2/5
1000/1000 [==============================] - 26s 26ms/step - loss: 0.6931 - accuracy: 0.5167 - val_loss: 0.7003 - val_accuracy: 0.5092
Epoch 3/5
1000/1000 [==============================] - 26s 26ms/step - loss: 0.6825 - accuracy: 0.5461 - val_loss: 0.6978 - val_accuracy: 0.5088
Epoch 4/5
1000/1000 [==============================] - 27s 27ms/step - loss: 0.6674 - accuracy: 0.5584 - val_loss: 0.7088 - val_accuracy: 0.5023
Epoch 5/5
1000/1000 [==============================] - 26s 26ms/step - loss: 0.6531 - accuracy: 0.5683 - val_loss: 0.7197 - val_accuracy: 0.5132


In [29]:
loss, acc = model.evaluate(np.array(X_test), np.array(y_test))

print(f"Test Accuracy: {acc:.2f}") 

313/313 [==============================] - 2s 7ms/step - loss: 0.7120 - accuracy: 0.5230
Test Accuracy: 0.52
